# 02. Pulizia, divisione dei dati e primo modello (baseline)
**Progetto:** HelpDesk AI · **Passo 1:** dati

In questo notebook applichiamo le decisioni nate dal notebook 01

## Cosa faremo:
1. Import e configurazione
2. Caricamento e pulizia difensiva
3. Trovare i quasi-duplicati
4. Dividere i dati in train, validation e test
5. Salvare la divisione
6. Un riferimento minimo: il modello "stupido"
7. Il primo modello: TF-IDF + regressione logistica
8. Quanto conta il *data leakage*? (esperimento)
9. Esperimenti di preprocessing
10. Cosa ha imparato il modello
11. Analisi degli errori
12. Salvataggio del modello
13. Riepilogo

# 1. Import e configurazione
Rispetto al notebook 1 aggiungiamo:
| Strumento | A cosa serve |
|---|---|
| `NearestNeighbors` | trovare i ticket molto simili tra loro (quasi-duplicati) |
| `scipy.sparse.csgraph.connected_components` | unire i ticket simili in **gruppi** |
| `StratifiedGroupKFold` | dividere i dati tenendo insieme i gruppi e mantenendo le proporzioni delle categorie |
| `Pipeline` | unire TF-IDF e classificatore in un unico oggetto |
| `DummyClassifier` | un modello "finto" che fa da riferimento minimo |
| `LogisticRegression` | il nostro primo classificatore |
| metriche di `sklearn.metrics` | accuratezza, F1, report per categoria, matrice di confusione |
| `joblib` | salvare il modello su disco |


In [10]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from scipy.sparse.csgraph import connected_components

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

In [11]:
# --- Riproducibilità ---
RANDOM_STATE = 42

# --- Percorsi ---
PROJECT_DIR = Path("..")
DATA_PATH = PROJECT_DIR / "data" / "raw" / "all_tickets_processed_improved_v3.csv"
SPLIT_PATH = PROJECT_DIR / "data" / "processed" / "tickets_split.csv"
MODELS_DIR = PROJECT_DIR / "models"

# --- Colonne ---
TEXT_COL = "Document"
LABEL_COL = "Topic_group"

# --- Stile dei grafici ---
TESTO_SECONDARIO = "#52514e"
GRIGLIA = "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRIGLIA,
    "xtick.color": TESTO_SECONDARIO,
    "ytick.color": TESTO_SECONDARIO,
})
pd.set_option("display.max_colwidth", 200)

# 2. Caricamento e pulizia
Anche se nel notebook 01 non abbiamo trovato valori mancanti, testi vuoti, duplicati o conflitti scriviamo comunque i passaggi di pulizia per:
- Robustezza: se il dataset venisse aggiornato con righe sporche, il notebook le gestirebbe da solo
- Trasparenza: stampiamo quante righe rimuove ogni passaggi

In [12]:
df = pd.read_csv(DATA_PATH)
n_iniziale = len(df)
print(f"Righe caricate: {n_iniziale}")

def rimuovi(df_corrente, maschera_da_tenere, descrizione):
    # Applica un filtro e stampa quante righe ha rimosso
    prima = len(df_corrente)
    df_nuovo = df_corrente[maschera_da_tenere].copy()
    print(f"    {descrizione:<40} -{prima - len(df_nuovo)}")
    return df_nuovo

df = rimuovi(df, df[TEXT_COL].notna(), "testo mancante")
df = rimuovi(df, df[LABEL_COL].notna(), "categoria mancante")
df[TEXT_COL] = df[TEXT_COL].str.strip()
df[LABEL_COL] = df[LABEL_COL].str.strip()
df = rimuovi(df, df[TEXT_COL] != "", "testo vuoto")
df = rimuovi(df, ~df.duplicated(), "duplicati esatti")
in_conflitto = df.groupby(TEXT_COL)[LABEL_COL].transform("nunique") > 1
df = rimuovi(df, ~in_conflitto, "testi con categorie in conflitto")

df = df.reset_index(drop=True)
print(f"\nRighe finali: {len(df)} (rimosse {n_iniziale - len(df)})")

Righe caricate: 47837
    testo mancante                           -0
    categoria mancante                       -0
    testo vuoto                              -0
    duplicati esatti                         -0
    testi con categorie in conflitto         -0

Righe finali: 47837 (rimosse 0)


# 3. Trovare i quasi-duplicati
Molti ticket sono email standard che potrebbero differire anche per un solo nome o una data. Se dividiamo i dati a caso, un ticket può finire nel train ed il suo "quasi-duplicato" nel test.

La soluzione: raggruppare e dividere per gruppi

1. Rappresentiamo ogni ticket con TF-IDF
2. Misuriamo la similarità del coseno tra ticket (1 se due testi usano le stesse parole, 0 se non hanno nulla in comune)
3. Colleghiamo i tikcet con similarità ≥ 0,9 (soglia usata anche in analisi pubblicate su questo dataset)
4. Ogni insieme di ticket collegati diventa un gruppo
5. Nello split, un gruppo intero finisce in un solo insieme (train, validation o test)

In [13]:
%%time 
# Per calcolare quanto impiega la cella

SOGLIA_SIMILARITA = 0.9

# 1. Rappresen0tazione TF-IDF di tutti i ticket
tfidf_gruppi = TfidfVectorizer(min_df=2)
X_tutti = tfidf_gruppi.fit_transform(df[TEXT_COL])

# 2. Per ogni ticket cerchiamo i vicini entro una certa distanza.
# Distanza del coseno = 1 - similarità, quindi similarità >=0.9 vuol dire distanza <=0.1
# Per confrontare tutti i ticket usiamo working_memory=256, così scikit-learn lo fa a blocchi di 256 MB (eseguibile anche su PC con poca RAM)
import sklearn
with sklearn.config_context(working_memory=256):
    vicini = NearestNeighbors(metric="cosine", radius=1 - SOGLIA_SIMILARITA, algorithm="brute")
    vicini.fit(X_tutti)
    grafo = vicini.radius_neighbors_graph(X_tutti, mode="connectivity")

# 3. Ogni componente collegato del grafo è un gruppo di ticket simili
n_gruppi, etichetta_gruppo = connected_components(grafo, directed=False)
df["gruppo"] = etichetta_gruppo

dimensioni = df["gruppo"].value_counts()
nei_gruppi = dimensioni[dimensioni > 1]
print(f"Gruppi totali:                          {n_gruppi}")
print(f"Gruppi con più di un ticket:            {len(nei_gruppi)}")
print(f"Ticket che hanno almeno un 'gemello':   {nei_gruppi.sum()} ({nei_gruppi.sum() / len(df) * 100:.1f}%)")
print(f"Gruppo più grande:                      {dimensioni.max()} ticket")

Gruppi totali:                          42230
Gruppi con più di un ticket:            1643
Ticket che hanno almeno un 'gemello':   7250 (15.2%)
Gruppo più grande:                      648 ticket
CPU times: total: 23.9 s
Wall time: 24.4 s


In [14]:
# Mostriamo alcuni ticket del gruppo più grande
gruppo_grande = dimensioni.index[0]
display(df.loc[df['gruppo'] == gruppo_grande, [TEXT_COL, LABEL_COL]].head(5))

# Quante categorie diverse ci sono dentro ai gruppi? 
categorie_per_gruppo = df[df['gruppo'].isin(nei_gruppi.index)].groupby('gruppo')[LABEL_COL].nunique()
print(f"Gruppi con ticket di categorie diverse: {(categorie_per_gruppo > 1).sum()} su {len(nei_gruppi)}")

,Document,Topic_group
25,new purchase po purchase po dear purchased has please log allocation after item please take consideration mandatory receipts section order receive item ordered how video link please receipt number...,Purchase
32,new purchase po purchase po dear purchased please log allocation after item please take consideration mandatory receipts section order receive item ordered how video link please receipt number ite...,Purchase
36,new purchase po purchase po dear purchased has please log allocation please log retrieve old device after receive item please take consideration mandatory receipts section order make receipt item ...,Purchase
41,new purchase po purchase po dear purchased matte display please log allocation after receive item please take consideration mandatory receipts section order make receipt item ordered how video lin...,Purchase
53,new purchase po purchase po dear purchased monitor please log allocation after item please take consideration mandatory receipts section order receive item ordered how video link please receipt nu...,Purchase


Gruppi con ticket di categorie diverse: 171 su 1643


# Osservazioni (parte 3)
- **7.250 ticket (15,2%)** hanno almeno un quasi-gemello, in **1.643 gruppi**: in linea con le analisi pubblicate
  su questo dataset (~7.200 ticket in ~1.640 gruppi con la stessa soglia). Uno split casuale avrebbe causato data leakage.
- Il gruppo più grande ha **648 ticket**: tutte email standard di ordini d'acquisto (Purchase), cioè più di un quarto
  della categoria. Le prestazioni su Purchase saranno in parte "facilitate" dal testo standard.
- **171 gruppi su 1.643** contengono categorie diverse: in parte rumore nelle etichette, in parte "effetto catena"
  del raggruppamento (A simile a B, B simile a C).

# 4. Divisione dei dati in Train, Validation e Test
Train 80%
Validation 10%
Test 10%

Per dividere usiamo **`StratifiedGroupKFold`** che rispetta due vincoli insieme:
- **stratificato**: ogni insieme ha più o meno le stesse proporzioni di categorie del dataset
- **per gruppi**: i ticket dello stesso gruppo restano nello stesso insieme

Per evitare che un gruppo grande finisca nel test, proviamo tutte e 10 le parti come possibili e teniamo quella con le proporzioni di categorie più vicine a quelle del dataset completo. Così i gruppi più grandi finiranno nel train. Faremo poi lo stesso per la validation.

In [15]:
def scegli_divisione_bilanciata(dati, n_splits):
    divisore = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    proporzioni_totali = dati[LABEL_COL].value_counts(normalize=True)

    migliore, scarto_minimo = None, np.inf
    for idx_grande, idx_piccolo in divisore.split(dati, dati[LABEL_COL], groups=dati['gruppo']):
        proporzioni = dati[LABEL_COL].iloc[idx_piccolo].value_counts(normalize=True)
        proporzioni = proporzioni.reindex(proporzioni_totali.index, fill_value=0)
        scarto = (proporzioni - proporzioni_totali).abs().max() # differenza peggiore tra le categorie
        if scarto < scarto_minimo:
            migliore, scarto_minimo = (idx_grande, idx_piccolo), scarto
    return migliore, scarto_minimo

# --- Primo taglio (TEST) ---
(idx_resto, idx_test), scarto_test = scegli_divisione_bilanciata(df, n_splits=10)
df_resto = df.iloc[idx_resto]

# --- Secondo taglio (VALIDATION) ---
(idx_train_rel, idx_val_rel), scarto_val = scegli_divisione_bilanciata(df_resto, n_splits=9)
print(f"Scarto massimo di proporzione: test {scarto_test * 100:.1f} punti, validation {scarto_val * 100:.1f} punti")

# Gli indici del secondo taglio sono relativi a df_resto: li riportiamo a df
idx_train = df_resto.index[idx_train_rel]
idx_val = df_resto.index[idx_val_rel]

df["split"] = ""
df.loc[idx_train, "split"] = "train"
df.loc[idx_val, "split"] = "val"
df.loc[df.index[idx_test], "split"] = "test"

print(df["split"].value_counts())

Scarto massimo di proporzione: test 0.9 punti, validation 1.0 punti
split
train    38359
test      4739
val       4739
Name: count, dtype: int64


In [17]:
# --- Controllo 1: nessun gruppo deve comparire in due insiemi diversi ---
insiemi_per_gruppo = df.groupby("gruppo")["split"].nunique()
assert (insiemi_per_gruppo == 1).all(), "Errore: alcuni gruppi sono divisi tra più insiemi!"
print("OK: ogni gruppo si trova in un solo insieme.")

# --- Controllo 2: le proporzioni delle categorie sono simili? ---
proporzioni = pd.crosstab(df[LABEL_COL], df["split"], normalize="columns").mul(100).round(1)
proporzioni["dataset completo"] = (df[LABEL_COL].value_counts(normalize=True) * 100).round(1)
display(proporzioni[["train", "val", "test", "dataset completo"]])

OK: ogni gruppo si trova in un solo insieme.


split,train,val,test,dataset completo
Topic_group,,,,
Access,14.9,15.0,15.0,14.9
Administrative rights,3.7,3.7,3.7,3.7
HR Support,22.8,23.0,23.0,22.8
Hardware,28.4,28.7,28.7,28.5
Internal Project,4.4,4.5,4.5,4.4
Miscellaneous,14.7,14.9,14.9,14.8
Purchase,5.4,4.3,4.3,5.2
Storage,5.8,5.8,5.8,5.8


# 5. Salvataggio divisione
Salviamo il dataset con le colonne **gruppo** e **split** in data/processe/. In questo modo, nei prossimi passi, utilizzeremo la stessa divisione

In [18]:
df.to_csv(SPLIT_PATH, index=False)
print("Salvato:", SPLIT_PATH.resolve())

# Per comodità per il resto del notebook
train = df[df["split"] == "train"]
val = df[df["split"] == "val"]
X_train, y_train = train[TEXT_COL], train[LABEL_COL]
X_val, y_val = val[TEXT_COL], val[LABEL_COL]
# Il TEST non sarà necessario per ora

Salvato: C:\Users\logic\Desktop\PROGRAMMAZIONE\GitHub\Capstone_Project\data\processed\tickets_split.csv
